In [14]:
# ==============================
# 1. IMPORT LIBRARIES
# ==============================
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler

from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
import numpy as np


from sklearn.metrics import r2_score, mean_squared_error


In [15]:
# ==============================
# 2. LOAD DATA
# ==============================
df = pd.read_csv("insurance_pre.csv")

# Change target if needed
X = df.drop("charges", axis=1)
y = df["charges"]


In [16]:
# Convert categorical columns into numeric
X = pd.get_dummies(X, drop_first=True).astype(int)

In [17]:
# ==============================
# 3. TRAIN TEST SPLIT
# ==============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [18]:

# ==============================
# 4. SCALING (ONLY FOR SVM)
# ==============================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [19]:
# ==============================
# 5. SVM + GRIDSEARCH
# ==============================
param_grid_svm = [
    {'kernel': ['linear'], 'C': [1, 10, 100]},
    {'kernel': ['rbf'], 'C': [1, 10, 100], 'gamma': ['scale', 'auto']},
    {'kernel': ['poly'], 'C': [1, 10], 'degree': [2, 3], 'gamma': ['scale']},
]

grid_svm = GridSearchCV(SVR(), param_grid_svm, cv=5, n_jobs=-1, verbose=2)
grid_svm.fit(X, y)

print("\nBest SVM Params:", grid_svm.best_params_)


Fitting 5 folds for each of 13 candidates, totalling 65 fits

Best SVM Params: {'C': 100, 'kernel': 'linear'}


In [20]:
# ==============================
# 6. DECISION TREE + GRIDSEARCH
# ==============================
param_grid_dt = {
    'criterion': ['squared_error', 'absolute_error'],
    'max_features': ['log2', 'sqrt', None],
    'splitter': ['best', 'random']
}

grid_dt = GridSearchCV(DecisionTreeRegressor(), param_grid_dt, cv=5, n_jobs=-1, verbose=2)
grid_dt.fit(X_train, y_train)

print("\nBest Decision Tree Params:", grid_dt.best_params_)


Fitting 5 folds for each of 12 candidates, totalling 60 fits

Best Decision Tree Params: {'criterion': 'absolute_error', 'max_features': None, 'splitter': 'random'}


In [21]:
# ==============================
# 7. RANDOM FOREST + GRIDSEARCH
# ==============================
param_grid_rf = {
    'n_estimators': [10, 100],
    'max_features': ['sqrt', 'log2'],
    'criterion': ['squared_error', 'absolute_error']
}

grid_rf = GridSearchCV(RandomForestRegressor(), param_grid_rf, cv=5, n_jobs=-1, verbose=2)
grid_rf.fit(X_train, y_train)

print("\nBest Random Forest Params:", grid_rf.best_params_)


Fitting 5 folds for each of 8 candidates, totalling 40 fits

Best Random Forest Params: {'criterion': 'absolute_error', 'max_features': 'sqrt', 'n_estimators': 100}


In [22]:
# ==============================
# 8. PREDICTIONS
# ==============================
svm_pred = grid_svm.predict(X_test_scaled)
dt_pred = grid_dt.predict(X_test)
rf_pred = grid_rf.predict(X_test)

C:\Users\SATHYABOOPALAN\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but SVR was fitted with feature names
  warnings.warn(


In [23]:
print("\n===== MODEL PERFORMANCE =====")

print("SVM R2 Score:", r2_score(y_test, svm_pred))
print("Decision Tree R2 Score:", r2_score(y_test, dt_pred))
print("Random Forest R2 Score:", r2_score(y_test, rf_pred))

print("\n===== RMSE =====")

print("SVM RMSE:", mean_squared_error(y_test, svm_pred, squared=False))
print("Decision Tree RMSE:", mean_squared_error(y_test, dt_pred, squared=False))
print("Random Forest RMSE:", mean_squared_error(y_test, rf_pred, squared=False))


===== MODEL PERFORMANCE =====
SVM R2 Score: -1.310166283814913
Decision Tree R2 Score: 0.7314476207463687
Random Forest R2 Score: 0.8508325672971998

===== RMSE =====


TypeError: got an unexpected keyword argument 'squared'